<a href="https://colab.research.google.com/github/rajeshsr92/Linuxfirsrt/blob/main/VDMC_RPA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
# ==============================
# Steps 1+2: Out of Scope (AL) + Hold reason (AM) +
# Work Queue (AN) + Last 30 days E/M (AO)
# Super-fast, 100% clean, output-only
# ==============================

!pip -q install pandas openpyxl

import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.styles import PatternFill

# ---------- Input files ----------
ATB_FILE   = "ATB.xlsx"
HOLD_FILE  = "HOLD Report.xlsx"
WORKQ_FILE = "Work que report.xlsx"
PROD_FILE  = "Last 30 Days production.xlsx"

# ---------- Output ----------
OUT_FILE = "ATB_Final_Output.xlsx"

# ---------- ATB Columns ----------
ATB_ENC  = "Encntr Number"
ATB_AR   = "Current A/R Balance"
ATB_HP   = "Current Health Plan"
ATB_FC   = "Current Fin Class"
ATB_MS   = "Medical Service"

# ---------- Hold Report ----------
HOLD_ENC = "Encntr Number"
HOLD_RS  = "HOLD_REASON"

# ---------- Work Queue ----------
WQ_ENC   = "Encntr Number"
WQ_STATE = "WORK_FLOW_STATE"

# ---------- Last 30 days production ----------
PROD_ENC = "Encntr Number"
PROD_EM  = "E/M"

# ---------- Excel letter columns ----------
COL_AL = "AL"   # Out of Scope
COL_AM = "AM"   # Hold Reason
COL_AN = "AN"   # Work Queue Status
COL_AO = "AO"   # E/M

# Normalize encounter number
def norm_enc(x):
    if pd.isna(x):
        return None
    s = str(x).strip()
    if s.endswith(".0"):
        s = s[:-2]
    return s

# ==============================
# LOAD ALL FILES
# ==============================

df_atb  = pd.read_excel(ATB_FILE, engine="openpyxl")
df_hold = pd.read_excel(HOLD_FILE, engine="openpyxl")
df_wq   = pd.read_excel(WORKQ_FILE, engine="openpyxl")
df_prod = pd.read_excel(PROD_FILE, engine="openpyxl")

# Strip headers
df_atb.columns  = df_atb.columns.str.strip()
df_hold.columns = df_hold.columns.str.strip()
df_wq.columns   = df_wq.columns.str.strip()
df_prod.columns = df_prod.columns.str.strip()

# Normalize encounter numbers
df_atb["_enc"]  = df_atb[ATB_ENC].map(norm_enc)
df_hold["_enc"] = df_hold[HOLD_ENC].map(norm_enc)
df_wq["_enc"]   = df_wq[WQ_ENC].map(norm_enc)
df_prod["_enc"] = df_prod[PROD_ENC].map(norm_enc)

# ==============================
# AL — OUT OF SCOPE LOGIC
# ==============================

ar        = pd.to_numeric(df_atb[ATB_AR], errors="coerce")
finclass  = df_atb[ATB_FC].astype(str).str.lower()
health    = df_atb[ATB_HP].astype(str).str.upper().str.strip()
medserv   = df_atb[ATB_MS].astype(str).str.upper().str.strip()

cond_ketamine = medserv.isin(["KETAMINE", "KETAMINE CONSULT", "SLS THERAPY"])
cond_credit   = ar < 0
cond_client   = finclass.str.contains("client", na=False)
cond_selfpay  = health.isin(["SELF PAY", "SELF PAY NO DISCOUNT"])

df_atb["_AL"] = np.select(
    [cond_ketamine, cond_credit, cond_client, cond_selfpay],
    ["Ketamine/SLS", "Credit Balance", "Client Fin class", "Self Pay"],
    default=""
)

# ==============================
# AM — HOLD REASONS (EXPLODE)
# ==============================

def unique_list(series):
    seen = set()
    lst = []
    for v in series.dropna().astype(str).str.strip():
        if v not in seen:
            seen.add(v)
            lst.append(v)
    return lst

df_hold_group = (
    df_hold.groupby("_enc")[HOLD_RS]
           .apply(unique_list)
           .reset_index()
)

df_merge = df_atb.merge(df_hold_group, how="left", on="_enc")

df_merge[HOLD_RS] = df_merge[HOLD_RS].apply(
    lambda x: x if isinstance(x, list) and len(x) > 0 else [None]
)

df_final = df_merge.explode(HOLD_RS, ignore_index=True)

df_final["_AM"] = df_final[HOLD_RS]
df_final["_dup"] = df_final.duplicated(subset=["_enc"], keep="first")

# AR = 0 for newly duplicated rows
df_final.loc[df_final["_dup"] == True, ATB_AR] = 0

# ==============================
# AN — WORK QUEUE (ONLY ALLOWED STATES)
# ==============================

ALLOWED = [
    "At Risk Claim",
    "Past Due",
    "Pending Reimbursement Claim",
    "Technical Denial"
]

df_wq[WQ_STATE] = df_wq[WQ_STATE].astype(str).str.strip()

df_wq_f = df_wq[df_wq[WQ_STATE].isin(ALLOWED)].copy()
df_wq_f["__order"] = pd.Categorical(df_wq_f[WQ_STATE], ALLOWED, ordered=True)

df_wq_best = (
    df_wq_f.sort_values("__order")
           .drop_duplicates("_enc", keep="first")
           [["_enc", WQ_STATE]]
)

df_final = df_final.merge(df_wq_best, how="left", on="_enc")
df_final["_AN"] = df_final[WQ_STATE]

# ==============================
# AO — LAST 30 DAYS E/M
# ==============================

df_prod_best = (
    df_prod.dropna(subset=[PROD_EM])
           .drop_duplicates("_enc", keep="first")
           [["_enc", PROD_EM]]
)

df_final = df_final.merge(df_prod_best, how="left", on="_enc")
df_final["_AO"] = df_final[PROD_EM]

# ==============================
# WRITE BASE TABLE TO EXCEL
# ==============================

# Keep original ATB columns in the body
orig_cols = [c for c in df_atb.columns if c not in ["_enc", "_AL"]]
df_body = df_final[orig_cols].copy()

df_body.to_excel(OUT_FILE, index=False)

# ==============================
# POST-PROCESS: Add AL, AM, AN, AO + highlight duplicates
# ==============================

wb = load_workbook(OUT_FILE)
ws = wb.active

ws[f"{COL_AL}1"] = "Out of Scope"
ws[f"{COL_AM}1"] = "Hold reason"
ws[f"{COL_AN}1"] = "Work Queue report"
ws[f"{COL_AO}1"] = "Last 30 days action code"

yellow = PatternFill(start_color="FFFFF2CC", end_color="FFFFF2CC", fill_type="solid")

n = len(df_final)
for i in range(n):
    r = i + 2
    ws[f"{COL_AL}{r}"] = df_final["_AL"].iloc[i]
    ws[f"{COL_AM}{r}"] = df_final["_AM"].iloc[i]
    ws[f"{COL_AN}{r}"] = df_final["_AN"].iloc[i]
    ws[f"{COL_AO}{r}"] = df_final["_AO"].iloc[i]

    if df_final["_dup"].iloc[i]:
        for c in range(1, ws.max_column + 1):
            ws.cell(r, c).fill = yellow

wb.save(OUT_FILE)

print("✅ COMPLETED SUCCESSFULLY")
print(f"Output file created: {OUT_FILE}")
print("AL = Out of Scope (Ketamine/SLS > Credit Balance > Client Fin class > Self Pay)")
print("AM = Hold Reason (duplicates created, AR=0, highlighted)")
print("AN = Work Queue matching allowed statuses only")
print("AO = E/M from Last 30 Days production")

✅ COMPLETED SUCCESSFULLY
Output file created: ATB_Final_Output.xlsx
AL = Out of Scope (Ketamine/SLS > Credit Balance > Client Fin class > Self Pay)
AM = Hold Reason (duplicates created, AR=0, highlighted)
AN = Work Queue matching allowed statuses only
AO = E/M from Last 30 Days production


In [9]:
import pandas as pd
from datetime import datetime

# Load files
atb = pd.read_excel("ATB_Final_Output.xlsx")
wq = pd.read_excel("Work que report.xlsx")

# Rename headers
atb.rename(columns={
    'AP': 'Current Reason',
    'AQ': 'Scope Reason',
    'AR': 'Scope'
}, inplace=True)

# Out-of-scope values
out_scope_values = [
    "Client Fin class",
    "Credit Balance",
    "Ketamine/SLS",
    "Self Pay"
]

# Apply Out-of-Scope logic
mask_out = atb['Out of Scope'].isin(out_scope_values)
atb.loc[mask_out, 'Current Reason'] = atb.loc[mask_out, 'Out of Scope']
atb.loc[mask_out, 'Scope Reason'] = atb.loc[mask_out, 'Out of Scope']
atb.loc[mask_out, 'Scope'] = "Client"

# ---------------------------------------------------------
# Merge ONLY to bring CREATED_DATE for Days calculation
# ---------------------------------------------------------
merged = atb.merge(
    wq[['Encntr Number', 'CREATED_DATE']],
    on='Encntr Number',
    how='left'
)

# Calculate days difference from CREATED_DATE
today = datetime.today()
merged['Days_Old'] = (today - merged['CREATED_DATE']).dt.days

# If CREATED_DATE missing → set Days_Old = 999 (forces Transmitted 30+)
merged['Days_Old'] = merged['Days_Old'].fillna(999)

# ---------------------------------------------------------
# Use Work Queue report column from ATB file
# ---------------------------------------------------------
mask_pending = merged['Work Queue report'] == "Pending Reimbursement Claim"

# A: Transmitted 0–30  (but NOT 30 exactly)
cond_A = mask_pending & (merged['Days_Old'] < 30) & (merged['Current A/R Balance'] < 5000)
merged.loc[cond_A, 'Current Reason'] = "Transmitted 0-30"

# B: Transmitted 0–15 HD
cond_B = mask_pending & (merged['Days_Old'] < 15) & (merged['Current A/R Balance'] > 5000)
merged.loc[cond_B, 'Current Reason'] = "Transmitted 0-15 HD"

# C: Transmitted 15–30 HD (but NOT 30 exactly)
cond_C = mask_pending & (merged['Days_Old'].between(15, 29)) & (merged['Current A/R Balance'] > 5000)
merged.loc[cond_C, 'Current Reason'] = "Transmitted 15-30 HD"

# D: Transmitted 30+ (includes EXACT 30 days)
cond_D = mask_pending & (merged['Days_Old'] >= 30)
merged.loc[cond_D, 'Current Reason'] = "Transmitted 30+"

# ---------------------------------------------------------
# Update Scope Reason and Scope based on Current Reason
# ---------------------------------------------------------

merged.loc[merged['Current Reason'] == "Transmitted 0-15 HD", ['Scope Reason', 'Scope']] = [
    "Future Workable", "R1_AR"
]

merged.loc[merged['Current Reason'] == "Transmitted 0-30", ['Scope Reason', 'Scope']] = [
    "Future Workable", "R1_AR"
]

merged.loc[merged['Current Reason'] == "Transmitted 15-30 HD", ['Scope Reason', 'Scope']] = [
    "Current Workable", "R1_AR"
]

merged.loc[merged['Current Reason'] == "Transmitted 30+", ['Scope Reason', 'Scope']] = [
    "Current Workable", "R1_AR"
]

# Save new output file
merged.to_excel("ATB_Final_Output_Updated_Final.xlsx", index=False)
